# Plotting and Visualization (Part - 2)
## DAT540 Introduction to Data Science
## University of Stavanger
### L13.2

- **Plotting with pandas and seaborn**
- pandas itself has built-in methods that simplify creating visualizations from DataFrames and Series objects
- *seaborn* is an additional statistical graphics library that simplifies creating various common visualization
- It allows aggregation and summarization with plotting


- *Line Plots*
- Series and DataFrame each have a *plot* argument for making some basic plot types.
- By default *plot()* makes line plots

- A Series object's index is passed to matplotlib for plotting on the x-axis. It can be disabled by passing argument *use_index=False* to plot()
- The x-axis ticks and limits can be adjusted with *xticks* and *xlim* options
- The y-axis respectively with *y-ticks* and *y-lim*
- *plot* options/arguments for Series / DataFrames

<img src='./images/series_plot.png'>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sbn 
# install seaborn in your environment if you don't have it , Run pip install seaborn in your vscode terminal or
# run !pip install seaborn in your jupyter notebook

In [ ]:
s = pd.Series(np.random.randn(10).cumsum(), index=np.arange(0, 100, 10))
s.plot()

- A DataFrame's plot method plots each of its columns as a different line on the same subplot, creating a legend automatically
- Like Series, a DataFrame plot instance creates a line plot by default and is equivalent to *df.plot.line()*
- DataFrame has a number of additional options allowing more flexibility with how the columns are handled
  - e.g. whether to plot them all on same subplot or to create seperate subplots
  
<img src='images/dataframe_plot.png'>

In [ ]:
df = pd.DataFrame(np.random.randn(10, 4).cumsum(0),
                 columns=['A', 'B', 'C', 'D'],
                 index=np.arange(0, 100, 10))
df

In [ ]:
df.plot()

- *Bar Plots*
- *plot.bar()* and *plot.barh()* makes vertical and horizontal bar plots
- Series or DataFrame index will be used as the x (bar) or y (barh) ticks
- With DataFrame, bar plots groups the values in each row together in a group in bars, side by side, for each value
- A stacked bar plot can be created by passing stacked=True option to DataFrame plot

In [ ]:
fig, axes = plt.subplots(2, 1)
data = pd.Series(np.random.randn(16), index=list('abcdefghijklmnop'))
data.plot.bar(ax=axes[0], color='k', alpha=0.7)
data.plot.barh(ax=axes[1], color='k', alpha=0.7) # horizontal barplot

In [ ]:
df = pd.DataFrame(np.random.rand(6, 4),
                 index=['one', 'two', 'three', 'four', 'five', 'six'],
                 columns=pd.Index(['A', 'B', 'C', 'D'], name='Genus'))
df

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)
df.plot.bar(ax=ax)
ax.legend(loc='best')

In [ ]:
# Creating Stacked bargraphs
fig = plt.figure()
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2)
df.plot.bar(ax=ax1, stacked=True)
df.plot.barh(ax=ax2, stacked=True)
ax1.legend(loc='best')
ax2.legend(loc='best')
fig.subplots_adjust(wspace=.5, hspace=0)

In [ ]:
# Data exploration and feature engineering
titanic = pd.read_csv('./data/titanic.csv',
                      delimiter=',', index_col='PassengerId')
titanic.shape

In [ ]:
titanic.head(5)

In [ ]:
titanic.info()

In [ ]:
titanic.isnull().sum(0)

In [ ]:
titanic.shape

In [ ]:
# removing row with having Embarked value as Null. Only two such row present
titanic.dropna(subset=['Embarked'], inplace=True)
# We drop the cabin column as it has a lot of missing data and if we can interpolate it from the Fare or Ticket column, then we dont need it
titanic.drop(['Cabin'], axis=1, inplace=True)
titanic.shape

In [ ]:
# Extracting the title from the Name column
# The unique values of the title column
titanic['Name'].apply(lambda x: [sal.title() for sal in x.split(' ')] ).apply(lambda x: x[1]).unique()

In [ ]:
titanic['Name'].iloc[887]

In [ ]:

salutations = titanic['Name'].apply(lambda x: [sal.title() for sal in x.split(' ')] ).apply(lambda x: x[1]).unique()
titanic['Salutations'] = titanic['Name'].apply(lambda x: [sal.title() for sal in x.split(' ') if sal.title() in salutations][0] )


In [ ]:
# removing row with having Salutations value as Null. 
titanic.dropna(subset=['Salutations'], inplace=True)
titanic.shape

In [ ]:
titanic.head(10)

In [ ]:
# Get the mean age by salutation
meanAgeBySalutation = titanic[['Salutations', 'Age']].groupby('Salutations').transform('mean')
# Filling the missing Age values with the mean age of the corresponding salutation
titanic.loc[titanic['Age'].isnull(), 'Age'] = titanic.loc[titanic['Age'].isnull(), 'Age'].index.map(lambda x: meanAgeBySalutation.loc[x, 'Age']).to_numpy()

titanic.isnull().sum(0)

In [ ]:
# since now we have only one row where age is missing, we remove that row
titanic.dropna(subset=['Age'], inplace=True)

In [ ]:
# We drop the Salutations, we donot need it now
titanic.drop(['Salutations'], axis=1, inplace=True)
titanic.isnull().sum(0)

In [ ]:
# Visualizing the relation between Sex and Survived. Do we see any relevence
fig = plt.figure(figsize=(8,8))
ax= fig.add_subplot(1,1,1)
pd.crosstab(titanic['Survived'], titanic['Sex']).plot.bar(ax=ax)

- In exploratory data analysis scatter pair plots describes relationships among group of variables (a.k.a. *pairs* plot or *scatter plot matrix*). 
- *seaborn* has a convenient *pairplot* function, which supports histograms or density estimates of each variable along the diagonal
- *heatmap* top-level method of seaborn plots rectangular data as a color-encoded matrix.

In [ ]:
sbn.pairplot(data=titanic, 
              x_vars=['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'],
              y_vars=['Survived'],
            hue='Survived')

In [ ]:
# Lets see the correlations
fig = plt.figure(figsize=(5,5))
ax = fig.add_subplot(1,1,1)
corr = titanic[list(titanic.describe().columns)].corr()
#Generate Color Map, red & blue
colormap = sbn.diverging_palette(220, 10, as_cmap=True)
#Generate Heat Map, allow annotations and place floats in map
sbn.heatmap(corr, cmap=colormap, annot=True, fmt=".2f", ax=ax)

In [ ]:
# Pclass vs Survived. Since we have limited number of unique values for Pclass, we can represent it in a barplot
titanic['Pclass'].unique().shape

In [ ]:
sbn.barplot(data=titanic, x='Pclass', y='Survived')

In [ ]:
# Age vs Survived. Since we have a lot number of unique values for Age, we can represent it in density bins
titanic['Age'].unique().shape

In [ ]:
# First we look at the age data
fig = plt.figure(figsize=(6,6))
ax = fig.add_subplot(1, 1, 1)
titanic['Age'].plot.hist(ax=ax)

In [ ]:
# Overall age have very less relationship with surviving. 
# However, it might be that chucks of age have better possibility to survive than others
fig = plt.figure(figsize=(6,6))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2)
sbn.histplot(titanic['Age'][titanic['Survived'] == 0], ax=ax1)
sbn.histplot(titanic['Age'][titanic['Survived'] == 1], ax=ax2)
fig.subplots_adjust(wspace=.25, hspace=0)
_ = ax1.set_title('Survived=False')
_ = ax2.set_title('Survived=True')

In [ ]:
# We observe that age betweeen 0-5 has a peek and survives more often than not
fig = plt.figure(figsize=(50,4))
ax = fig.add_subplot(1, 2, 1)
sbn.kdeplot(titanic['Age'][titanic['Survived'] == 0], color='red', fill=True, ax=ax)
sbn.kdeplot(titanic['Age'][titanic['Survived'] == 1], color='green', fill=True, ax=ax)

In [ ]:
# SibSp vs Survived. Since we have limited number of unique values for SibSp, we can represent it in a barplot
titanic['SibSp'].unique().shape

In [ ]:
# SibSp vs Survived
# Single or people having one or two siblings have better chance to survive
sbn.barplot(data=titanic, x='SibSp', y='Survived')

In [ ]:
# Parch vs Survived. Since we have limited number of unique values for Parch, we can represent it in a barplot
titanic['Parch'].unique().shape

In [ ]:
# Parch vs Survived
# People in Parch 1 - 3 have better chance to survive
sbn.barplot(data=titanic, x='Parch', y='Survived')

In [ ]:
# Fare vs Survived. Since we have a lot number of unique values for Fare, we can represent it in  bins
# This will help us identify, which chunks of fares actually have some relationship to survive
titanic['Fare'].unique().shape

In [ ]:
# First we look at the fare data. It is skewed, so we transform it
fig = plt.figure(figsize=(4,4))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2)
titanic['Fare'].plot.hist(ax=ax1)
titanic['Fare'] = titanic['Fare'].map(lambda x: np.log(x) if x > 0 else 0)
titanic['Fare'].plot.hist(ax=ax2)
_ = ax1.set_title('Skewed (original)')
_ = ax2.set_title('logarithmic (transformed)')
fig.subplots_adjust(wspace=1, hspace=0)

In [ ]:
# We observe that fare between 3-6 have slightly better chance to survive. But overall they all have high corr.
fig = plt.figure(figsize=(50,4))
ax = fig.add_subplot(1, 2, 1)
sbn.kdeplot(titanic['Fare'][titanic['Survived'] == 0], color='red', fill=True, ax=ax)
sbn.kdeplot(titanic['Fare'][titanic['Survived'] == 1], color='green', fill=True, ax=ax)

In [ ]:
# Pclass vs Survived. Since we have limited number of unique values for Pclass, we can represent it in a barplot
titanic['Pclass'].unique().shape

In [ ]:
# Pclass vs Survived
sbn.barplot(data=titanic, x='Pclass', y='Survived')

In [ ]:
# Embarked vs Survived. Since we have limited number of unique values for Embarked, we can represent it in a barplot
titanic['Embarked'].unique().shape

In [ ]:
# Embarked vs Survived
# People who Embarked at C have better chance, can people from that port C swim better?
sbn.barplot(data=titanic, x='Embarked', y='Survived')

In [ ]:
fig = plt.figure(figsize=(5,5))
ax1 = fig.add_subplot(2,2,1)
ax2 = fig.add_subplot(2,2,2)
ax3 = fig.add_subplot(2,2,3)
ax4 = fig.add_subplot(2,2,4)

# We know that female have better chance of surviving than man
pd.crosstab(titanic['Embarked'], titanic['Sex']).plot.bar(ax=ax1) 
# We know that chance of surviving in class 1> 2> 3
pd.crosstab(titanic['Embarked'], titanic['Pclass']).plot.bar(ax=ax2)
# We know that people with 0 - 2 siblings slightly have better chance of surviving
pd.crosstab(titanic['Embarked'], titanic['SibSp']).plot.bar(ax=ax3)
# We know that people in Parch 1 - 3 have better chance of surviving
pd.crosstab(titanic['Embarked'], titanic['Parch']).plot.bar(ax=ax4)
fig.subplots_adjust(wspace=.25, hspace=.25)

In [ ]:
# Let us create bins for age as AgeBins
# Age has a min of 0 and max of 80
titanic['AgeBins'] = pd.cut(titanic['Age'], range(0, 85, 5), right=True)

# We create dummies and bins. Why?
titanic = pd.concat([titanic, 
                     pd.get_dummies(titanic['Sex'], prefix='sex_'), 
                     pd.get_dummies(titanic['AgeBins'], prefix='age_'), 
                     pd.get_dummies(titanic['Embarked'], prefix='emb_'),
                     pd.get_dummies(titanic['Pclass'], prefix='pcl_'),
                     pd.get_dummies(titanic['SibSp'], prefix='sib_'),
                     pd.get_dummies(titanic['Parch'], prefix='par_')], axis=1)

In [ ]:
titanic.head()

In [ ]:
# Lets see the correlations again
fig = plt.figure(figsize=(50,50))
ax = fig.add_subplot(1,1,1)
corr = pd.concat([titanic[['Survived','Age', 'Fare']], titanic.select_dtypes(include=bool)], axis=1).corr()
#Generate Color Map, red & blue
colormap = sbn.diverging_palette(220, 10, as_cmap=True)
#Generate Heat Map, allow annotations and place floats in map
sbn.heatmap(corr, cmap=colormap, annot=True, fmt=".1f", ax=ax, annot_kws={"size": 20})

In [ ]:
# We choose the high correlated variables
threshold = 0.2
titanic_trans = titanic[np.append(corr['Survived'][1:][corr['Survived'][1:].map(lambda x: np.abs(x) > threshold)].index.values, ['Survived'])]
titanic_trans.head()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error as MAPE

In [ ]:
# We take 80% of the dataset to train and rest to test
# there are further optimization that can be performed on the data. This is just an initial example
from sklearn.model_selection import train_test_split

X = titanic_trans.drop('Survived', axis=1)
y = titanic_trans['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Recreate train and test DataFrames
train = pd.concat([X_train, y_train], axis=1)
test = pd.concat([X_test, y_test], axis=1)

In [ ]:
predictions = pd.DataFrame([])
model = LogisticRegression()
model.fit(train.loc[:, train.columns != 'Survived'], train['Survived'])
predictions['Survived'] = np.ceil(model.predict(test.loc[:, test.columns != 'Survived']))
predictions['PassengerId'] = test.index.values
predictions.set_index('PassengerId', inplace=True)
# since we are doing random sampling in choosing the train and test set, expect different MAPE across multiple runs
print('Accuracy:',  (1 - MAPE(test['Survived'], predictions['Survived'])) * 100) 

In [ ]:
%matplotlib inline
fig = plt.figure(figsize=(20,1))
ax = fig.add_subplot(1,1,1)
result = test[['Survived']].join(predictions, rsuffix='_pred').reset_index()
result.plot.scatter(ax=ax, x='PassengerId', y='Survived', color='green', marker='+', s=100)
result.plot.scatter(ax=ax, x='PassengerId', y='Survived_pred', color='red', marker='x', s=100)

##### Read more about improving the titanic survival dataset https://www.kaggle.com/yassineghouzam/titanic-top-4-with-ensemble-modeling
##### More about Seaborn https://seaborn.pydata.org/